<div style="background: linear-gradient(135deg, #0f0c29, #302b63, #24243e); padding: 40px 30px; border-radius: 16px; margin-bottom: 20px;">
<h1 style="text-align:center; color:#00cec9; font-family:Verdana; letter-spacing:3px;">
?? Image Matching Challenge 2022 ? EDA & CV Fundamentals
</h1>
<h4 style="text-align:center; color:#dfe6e9; font-family:Verdana;">
Notebook 1 of 2 ? Data Understanding & Computer Vision Techniques
</h4>
<hr style="border:1px solid #6c5ce7;">
<p style="color:#fab1a0; font-family:Verdana; font-size:14px; text-align:center;">
This notebook presents our end-to-end exploratory analysis for IMC 2022. We profile scene-level statistics, inspect covisibility behavior, and validate camera geometry before matching. The workflow is structured for direct Kaggle execution with reproducible setup, robust visual diagnostics, and implementation-ready CV foundations for downstream pose estimation.
</p>
</div>

## ?? Table of Contents

1. [Environment Setup & Imports](#sec1)
2. [Dataset Overview & Loading](#sec2)
3. [Exploratory Data Analysis](#sec3)
   - 3.1 Scene Distribution
   - 3.2 Covisibility Analysis
   - 3.3 Camera Intrinsics
   - 3.4 Camera Extrinsics (Translation & Rotation)
   - 3.5 Image Properties
4. [Covisibility Deep-Dive ? Real Image Pairs](#sec4)
5. [Computer Vision Fundamentals](#sec5)
   - 5.1 Feature Detection ? SIFT
   - 5.2 Feature Detection ? ORB
   - 5.3 Feature Detection ? AKAZE
   - 5.4 Detector Comparison (Stats & Visuals)
   - 5.5 Feature Matching ? Brute-Force + Ratio Test
   - 5.6 RANSAC Filtering
   - 5.7 Baseline CV Parity Check (SIFT vs ORB)
6. [Camera Calibration, Stereo Vision & Depth](#sec6)
   - 6.1 Chessboard Calibration & Undistortion
   - 6.2 Stereo Disparity & Depth on IMC Pair
7. [Fundamental & Essential Matrices](#sec7)
   - 7.1 SIFT-Based Epipolar Geometry
   - 7.2 ORB + RANSAC Epipolar Geometry
8. [SfM Pipeline Overview & Key Takeaways](#sec8)


<a id="sec1"></a>
## 1 · Environment Setup & Imports

In [ ]:
import os, gc, time, random, warnings, csv, math
import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import ListedColormap
from PIL import Image
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

# Reproducibility
def seed_it_all(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
seed_it_all()

# Standard matplotlib style for Kaggle compatibility
plt.style.use('default')

DATA_DIR = "/kaggle/input/image-matching-challenge-2022"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
print(f"? Data directory: {DATA_DIR}")
print(f"? OpenCV version: {cv2.__version__}")
print(f"? NumPy  version: {np.__version__}")


<a id="sec2"></a>
## 2 · Dataset Overview & Loading

In [ ]:
# ── Load scaling factors & discover scenes ──
scaling_df = pd.read_csv(os.path.join(TRAIN_DIR, "scaling_factors.csv"))
scale_map = scaling_df.groupby("scene")["scaling_factor"].first().to_dict()
TRAIN_SCENES = scaling_df.scene.unique().tolist()
print(f"🏛️  Found {len(TRAIN_SCENES)} scenes: {TRAIN_SCENES}")

# ── Build comprehensive train map ──
train_map = {}
for scene in tqdm(TRAIN_SCENES, desc="Loading scene metadata"):
    train_map[scene] = {}
    img_dir = os.path.join(TRAIN_DIR, scene, "images")
    train_map[scene]["images"] = sorted([
        os.path.join(img_dir, f) for f in os.listdir(img_dir) if f.endswith(".jpg")
    ])
    train_map[scene]["image_ids"] = [
        os.path.splitext(os.path.basename(p))[0] for p in train_map[scene]["images"]
    ]
    train_map[scene]["cal_df"] = pd.read_csv(os.path.join(TRAIN_DIR, scene, "calibration.csv"))
    train_map[scene]["cal_df"]["image_path"] = (
        img_dir + "/" + train_map[scene]["cal_df"]["image_id"] + ".jpg"
    )
    pco = pd.read_csv(os.path.join(TRAIN_DIR, scene, "pair_covisibility.csv"))
    pco[["image_id_1", "image_id_2"]] = pco["pair"].str.split("-", expand=True)
    pco["image_path_1"] = img_dir + "/" + pco["image_id_1"] + ".jpg"
    pco["image_path_2"] = img_dir + "/" + pco["image_id_2"] + ".jpg"
    train_map[scene]["pco_df"] = pco

print(f"\n✅ All {len(TRAIN_SCENES)} scenes loaded successfully.")

In [ ]:
# ── Combine into a single analysis DataFrame ──
_cal, _pco, _scene = [], [], []
for s in TRAIN_SCENES:
    m = train_map[s]
    _scene.append(pd.DataFrame({"scene": s, "f_path": m["images"]}))
    _cal.append(m["cal_df"])
    _pco.append(m["pco_df"])

all_pco = pd.concat(_pco, ignore_index=True)
all_cal = pd.concat(_cal, ignore_index=True)
train_df = pd.concat(_scene, ignore_index=True)
train_df["image_id"] = train_df.f_path.apply(lambda x: os.path.splitext(os.path.basename(x))[0])
train_df = train_df.merge(all_cal, on="image_id").drop(columns=["image_path"])

# Add mean covisibility per image
cov = pd.concat([
    all_pco[["image_id_1","covisibility"]].rename(columns={"image_id_1":"image_id"}),
    all_pco[["image_id_2","covisibility"]].rename(columns={"image_id_2":"image_id"}),
])
img_cov = cov.groupby("image_id")["covisibility"].mean().to_dict()
train_df["mean_covisibility"] = train_df.image_id.map(img_cov)

# Parse intrinsics
train_df["fx"] = train_df.camera_intrinsics.apply(lambda x: float(x.split()[0]))
train_df["fy"] = train_df.camera_intrinsics.apply(lambda x: float(x.split()[4]))
train_df["x0"] = train_df.camera_intrinsics.apply(lambda x: float(x.split()[2]))
train_df["y0"] = train_df.camera_intrinsics.apply(lambda x: float(x.split()[5]))
train_df["s"]  = train_df.camera_intrinsics.apply(lambda x: float(x.split()[1]))

# Parse extrinsics
for i, name in enumerate(["t_x","t_y","t_z"]):
    train_df[name] = train_df.translation_vector.apply(lambda x, idx=i: float(x.split()[idx]))

print(f"✅ Combined DataFrame shape: {train_df.shape}")
print(f"   fx == fy for all? {(train_df.fx == train_df.fy).all()}")
print(f"   All skew = 0?     {(train_df.s == 0).all()}")
train_df.head()

### Helper Functions

In [ ]:
def arr_from_str(s):
    return np.fromstring(s, sep=" ").reshape(-1, 3).squeeze()

def plot_image_pair(path1, path2, titles=None, figsize=(20, 8)):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    for ax, p, t in zip(axes, [path1, path2], titles or [path1, path2]):
        img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(t, fontweight="bold", fontsize=12, color="#00cec9")
        ax.axis("off")
    fig.patch.set_facecolor('#1a1a2e')
    plt.tight_layout()
    plt.show()

def build_id2cal(train_map, scenes):
    m = {}
    for s in scenes:
        for _, r in train_map[s]["cal_df"].iterrows():
            iid = r["image_id"]; m[iid] = {}
            m[iid]["K"] = arr_from_str(r["camera_intrinsics"])
            m[iid]["R"] = arr_from_str(r["rotation_matrix"])
            m[iid]["T"] = arr_from_str(r["translation_vector"])
            m[iid]["path"] = r["image_path"]
    return m

id2cal = build_id2cal(train_map, TRAIN_SCENES)
print(f"✅ Calibration map built for {len(id2cal)} images")

<a id="sec3"></a>
## 3 · Exploratory Data Analysis

### 3.1 Scene Distribution

In [ ]:
scene_counts = train_df["scene"].value_counts().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(scene_counts.index, scene_counts.values, color="steelblue", edgecolor="black")
ax.set_title("Image Counts per Scene")
ax.set_xlabel("Scene")
ax.set_ylabel("Number of Images")
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


### 3.2 Covisibility Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.hist(train_df["mean_covisibility"].dropna(), bins=60, color="slateblue", edgecolor="black", alpha=0.8)
ax.set_title("Mean Covisibility Distribution")
ax.set_xlabel("Mean Covisibility")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

# Per-scene covisibility histograms
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
for idx, scene in enumerate(TRAIN_SCENES):
    ax = axes[idx // 4][idx % 4]
    cov_vals = train_map[scene]["pco_df"]["covisibility"].values
    ax.hist(cov_vals, bins=40, color="mediumpurple", edgecolor="black", alpha=0.8)
    ax.axvline(0.1, color="crimson", linestyle="--", linewidth=2, label="threshold=0.1")
    ax.set_title(scene.replace("_"," ").title(), fontsize=10)
    ax.set_xlabel("covisibility", fontsize=8)
    mean_cov = np.mean(cov_vals)
    ax.text(0.95, 0.95, f"mu={mean_cov:.3f}", transform=ax.transAxes,
            ha='right', va='top', fontsize=9)
fig.suptitle("Covisibility Distribution per Scene", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()


### 3.3 Camera Intrinsics

In [ ]:
for col, label in [("x0","Principal Point x"), ("y0","Principal Point y"), ("fx","Focal Length")]:
    fig, ax = plt.subplots(figsize=(12, 4))
    vals = train_df[col].dropna().values
    if col == "fx":
        vals = vals[vals > 0]
        ax.set_xscale("log")
    ax.hist(vals, bins=50, color="cornflowerblue", edgecolor="black", alpha=0.85)
    ax.set_title(f"{label} Distribution")
    ax.set_xlabel(f"{label}{' (log scale)' if col == 'fx' else ''}")
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.show()


### 3.4 Camera Extrinsics

In [ ]:
for col, label in [("t_x","Translation X"), ("t_y","Translation Y"), ("t_z","Translation Z")]:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.hist(train_df[col].dropna(), bins=60, color="seagreen", edgecolor="black", alpha=0.85)
    ax.set_yscale("log")
    ax.set_title(f"{label} Distribution")
    ax.set_xlabel(label)
    ax.set_ylabel("Count (log)")
    plt.tight_layout()
    plt.show()


### 3.5 Image Properties

In [ ]:
# Sample images to check dimensions
sample_sizes = []
for scene in TRAIN_SCENES[:4]:
    for p in train_map[scene]["images"][:20]:
        img = cv2.imread(p)
        if img is not None:
            h, w = img.shape[:2]
            sample_sizes.append({"scene": scene, "height": h, "width": w,
                                 "aspect": round(w/h, 2), "longest": max(h,w)})
sz_df = pd.DataFrame(sample_sizes)

fig, ax = plt.subplots(figsize=(10, 6))
for scene, g in sz_df.groupby("scene"):
    ax.scatter(g["width"], g["height"], s=45, alpha=0.7, label=scene)
ax.set_title("Image Dimensions (sampled)")
ax.set_xlabel("Width")
ax.set_ylabel("Height")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("
?? Image dimension statistics:")
print(sz_df[["height","width","aspect","longest"]].describe().round(2))


<a id="sec4"></a>
## 4 · Covisibility Deep-Dive — Real Image Pairs

Visualising high and low covisibility pairs from real dataset images.

In [ ]:
# High covisibility example
scene = "british_museum"
pco = train_map[scene]["pco_df"]
high_row = pco.sort_values("covisibility", ascending=False).iloc[0]
print(f"🔗 HIGH covisibility = {high_row.covisibility:.4f} ('{scene}')")
plot_image_pair(high_row.image_path_1, high_row.image_path_2,
    [f"Image 1 — COV={high_row.covisibility:.4f}",
     f"Image 2 — COV={high_row.covisibility:.4f}"])

# Low covisibility example
low_row = pco[pco.covisibility < 0.05].iloc[0]
print(f"\n🔗 LOW covisibility = {low_row.covisibility:.4f} ('{scene}')")
plot_image_pair(low_row.image_path_1, low_row.image_path_2,
    [f"Image 1 — COV={low_row.covisibility:.4f}",
     f"Image 2 — COV={low_row.covisibility:.4f}"])

In [ ]:
# One random pair per scene
print("📷 Random image pairs across all scenes:\n")
for scene in TRAIN_SCENES:
    row = train_map[scene]["pco_df"].sample(1, random_state=42).iloc[0]
    plot_image_pair(row.image_path_1, row.image_path_2,
        [f"{scene} — Img1 · COV={row.covisibility:.3f}",
         f"{scene} — Img2 · COV={row.covisibility:.3f}"])

<a id="sec5"></a>
## 5 · Computer Vision Fundamentals

### 5.1 Feature Detection — SIFT

In [ ]:
# Use real dataset images (Brandenburg Gate)
IMG1_PATH = os.path.join(TRAIN_DIR, "brandenburg_gate/images/00883281_9633489441.jpg")
IMG2_PATH = os.path.join(TRAIN_DIR, "brandenburg_gate/images/01069771_8567470929.jpg")

img1_gray = cv2.imread(IMG1_PATH, cv2.IMREAD_GRAYSCALE)
img2_gray = cv2.imread(IMG2_PATH, cv2.IMREAD_GRAYSCALE)
img1_rgb = cv2.cvtColor(cv2.imread(IMG1_PATH), cv2.COLOR_BGR2RGB)
img2_rgb = cv2.cvtColor(cv2.imread(IMG2_PATH), cv2.COLOR_BGR2RGB)

# SIFT detection
sift = cv2.SIFT_create(nfeatures=2000)
t0 = time.time()
kp1_sift, des1_sift = sift.detectAndCompute(img1_gray, None)
kp2_sift, des2_sift = sift.detectAndCompute(img2_gray, None)
sift_time = time.time() - t0

# Draw rich keypoints
sift_vis1 = cv2.drawKeypoints(img1_rgb, kp1_sift, None,
    color=(0,255,200), flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
sift_vis2 = cv2.drawKeypoints(img2_rgb, kp2_sift, None,
    color=(0,255,200), flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
axes[0].imshow(sift_vis1); axes[0].set_title(f"SIFT — Image 1 ({len(kp1_sift)} kps)", color="#00cec9")
axes[1].imshow(sift_vis2); axes[1].set_title(f"SIFT — Image 2 ({len(kp2_sift)} kps)", color="#00cec9")
for ax in axes: ax.axis("off")
fig.suptitle(f"SIFT Keypoints · Time: {sift_time:.3f}s · Descriptor dim: 128-D float",
             fontsize=14, color="#fdcb6e")
plt.tight_layout(); plt.show()

# Keypoint heatmap
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for ax, kps, title in zip(axes, [kp1_sift, kp2_sift], ["Image 1", "Image 2"]):
    x = [k.pt[0] for k in kps]; y = [k.pt[1] for k in kps]
    ax.hexbin(x, y, gridsize=30, cmap='magma', mincnt=1)
    ax.set_title(f"SIFT Keypoint Density — {title}", color="#00cec9")
    ax.invert_yaxis(); ax.axis("off")
plt.tight_layout(); plt.show()

print(f"\n📊 SIFT Statistics:")
print(f"   Image 1: {len(kp1_sift)} keypoints")
print(f"   Image 2: {len(kp2_sift)} keypoints")
print(f"   Descriptor shape: {des1_sift.shape}")
print(f"   Detection time: {sift_time:.4f}s")
responses1 = [k.response for k in kp1_sift]
print(f"   Keypoint response — mean: {np.mean(responses1):.4f}, max: {np.max(responses1):.4f}")

### 5.2 Feature Detection — ORB

In [ ]:
orb = cv2.ORB_create(nfeatures=2000)
t0 = time.time()
kp1_orb, des1_orb = orb.detectAndCompute(img1_gray, None)
kp2_orb, des2_orb = orb.detectAndCompute(img2_gray, None)
orb_time = time.time() - t0

orb_vis1 = cv2.drawKeypoints(img1_rgb, kp1_orb, None, color=(255,100,50), flags=0)
orb_vis2 = cv2.drawKeypoints(img2_rgb, kp2_orb, None, color=(255,100,50), flags=0)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
axes[0].imshow(orb_vis1); axes[0].set_title(f"ORB — Image 1 ({len(kp1_orb)} kps)", color="#e17055")
axes[1].imshow(orb_vis2); axes[1].set_title(f"ORB — Image 2 ({len(kp2_orb)} kps)", color="#e17055")
for ax in axes: ax.axis("off")
fig.suptitle(f"ORB Keypoints · Time: {orb_time:.3f}s · Descriptor dim: 32-D binary",
             fontsize=14, color="#fdcb6e")
plt.tight_layout(); plt.show()

# Heatmaps
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for ax, kps, title in zip(axes, [kp1_orb, kp2_orb], ["Image 1","Image 2"]):
    x = [k.pt[0] for k in kps]; y = [k.pt[1] for k in kps]
    ax.hexbin(x, y, gridsize=30, cmap='inferno', mincnt=1)
    ax.set_title(f"ORB Keypoint Density — {title}", color="#e17055")
    ax.invert_yaxis(); ax.axis("off")
plt.tight_layout(); plt.show()

print(f"\n📊 ORB Statistics:")
print(f"   Image 1: {len(kp1_orb)} keypoints")
print(f"   Image 2: {len(kp2_orb)} keypoints")
print(f"   Descriptor shape: {des1_orb.shape}")
print(f"   Detection time: {orb_time:.4f}s")

### 5.3 Feature Detection — AKAZE

In [ ]:
akaze = cv2.AKAZE_create()
t0 = time.time()
kp1_ak, des1_ak = akaze.detectAndCompute(img1_gray, None)
kp2_ak, des2_ak = akaze.detectAndCompute(img2_gray, None)
akaze_time = time.time() - t0

ak_vis1 = cv2.drawKeypoints(img1_rgb, kp1_ak, None, color=(100,255,100),
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
ak_vis2 = cv2.drawKeypoints(img2_rgb, kp2_ak, None, color=(100,255,100),
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
axes[0].imshow(ak_vis1); axes[0].set_title(f"AKAZE — Image 1 ({len(kp1_ak)} kps)", color="#00b894")
axes[1].imshow(ak_vis2); axes[1].set_title(f"AKAZE — Image 2 ({len(kp2_ak)} kps)", color="#00b894")
for ax in axes: ax.axis("off")
fig.suptitle(f"AKAZE Keypoints · Time: {akaze_time:.3f}s",
             fontsize=14, color="#fdcb6e")
plt.tight_layout(); plt.show()

print(f"\n📊 AKAZE Statistics:")
print(f"   Image 1: {len(kp1_ak)} keypoints, desc shape: {des1_ak.shape}")
print(f"   Image 2: {len(kp2_ak)} keypoints")
print(f"   Detection time: {akaze_time:.4f}s")

### 5.4 Detector Comparison

In [ ]:
comp_data = {
    "Detector": ["SIFT","ORB","AKAZE"],
    "Keypoints Img1": [len(kp1_sift), len(kp1_orb), len(kp1_ak)],
    "Keypoints Img2": [len(kp2_sift), len(kp2_orb), len(kp2_ak)],
    "Desc Dim": [128, 32, des1_ak.shape[1]],
    "Time (s)": [round(sift_time,4), round(orb_time,4), round(akaze_time,4)],
}
comp_df = pd.DataFrame(comp_data)
print("
?? Feature Detector Comparison:")
print(comp_df.to_string(index=False))

labels = ["Img1 KPs", "Img2 KPs", "Desc Dim"]
x = np.arange(len(labels))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
for i, det in enumerate(comp_df["Detector"]):
    row = comp_df[comp_df.Detector == det].iloc[0]
    vals = [row["Keypoints Img1"], row["Keypoints Img2"], row["Desc Dim"]]
    ax.bar(x + (i - 1) * width, vals, width=width, label=det)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_title("Detector Comparison")
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(comp_df["Detector"], comp_df["Time (s)"], color=["teal", "tomato", "seagreen"], edgecolor="black")
ax.set_title("Detection Speed Comparison")
ax.set_ylabel("Time (s)")
plt.tight_layout()
plt.show()


### 5.5 Feature Matching — Brute-Force + Ratio Test

In [ ]:
# SIFT BF matching with ratio test
bf = cv2.BFMatcher(cv2.NORM_L2)
raw_matches = bf.knnMatch(des1_sift, des2_sift, k=2)
good_matches = [m for m, n in raw_matches if m.distance < 0.75 * n.distance]
good_matches = sorted(good_matches, key=lambda x: x.distance)

print(f"📊 SIFT BF Matching (Lowe ratio=0.75):")
print(f"   Raw kNN matches: {len(raw_matches)}")
print(f"   Good matches (after ratio test): {len(good_matches)}")
print(f"   Rejection rate: {100*(1 - len(good_matches)/len(raw_matches)):.1f}%")

# Visualise top matches
match_img = cv2.drawMatches(img1_rgb, kp1_sift, img2_rgb, kp2_sift,
    good_matches[:80], None,
    matchColor=(0, 255, 200), singlePointColor=(255, 100, 50),
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

fig, ax = plt.subplots(figsize=(22, 10))
ax.imshow(match_img)
ax.set_title(f"Top 80 SIFT Matches (BF + Ratio Test) — {len(good_matches)} total good matches",
             fontsize=14, color="#00cec9")
ax.axis("off")
plt.tight_layout(); plt.show()

# Match distance distribution
dists = [m.distance for m in good_matches]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(dists, bins=50, color="royalblue", edgecolor="black", alpha=0.85)
ax.set_title("Match Distance Distribution (Good Matches)")
ax.set_xlabel("L2 Distance")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()
print(f"   Distance ? mean: {np.mean(dists):.2f}, median: {np.median(dists):.2f}, min: {np.min(dists):.2f}")



### 5.6 RANSAC Filtering

In [ ]:
pts1 = np.float32([kp1_sift[m.queryIdx].pt for m in good_matches])
pts2 = np.float32([kp2_sift[m.trainIdx].pt for m in good_matches])

F, mask = cv2.findFundamentalMat(pts1, pts2, cv2.USAC_MAGSAC, 0.5, 0.999, 10000)
inlier_mask = mask.ravel().astype(bool)
inliers = [m for m, flag in zip(good_matches, inlier_mask) if flag]
outliers_count = len(good_matches) - len(inliers)

print(f"📊 RANSAC (USAC_MAGSAC) Results:")
print(f"   Input matches: {len(good_matches)}")
print(f"   Inliers: {len(inliers)}")
print(f"   Outliers removed: {outliers_count}")
print(f"   Inlier ratio: {len(inliers)/len(good_matches)*100:.1f}%")

# Draw inlier matches
inlier_img = cv2.drawMatches(img1_rgb, kp1_sift, img2_rgb, kp2_sift,
    inliers[:80], None,
    matchColor=(0, 255, 100), singlePointColor=(255, 50, 50),
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

fig, ax = plt.subplots(figsize=(22, 10))
ax.imshow(inlier_img)
ax.set_title(f"Matches After RANSAC — {len(inliers)} inliers (from {len(good_matches)} matches)",
             fontsize=14, color="#00b894")
ax.axis("off")
plt.tight_layout(); plt.show()

### 5.7 Baseline CV Parity Check (SIFT vs ORB)

This parity block mirrors the baseline comparison flow: timing, keypoint counts, and cross-check matching quality for SIFT and ORB.


In [ ]:
# Baseline parity check using cross-check matchers (same style as baseline assignment)
bf_sift_cc = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
bf_orb_cc = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

sift_cc_matches = sorted(bf_sift_cc.match(des1_sift, des2_sift), key=lambda m: m.distance)
orb_cc_matches = sorted(bf_orb_cc.match(des1_orb, des2_orb), key=lambda m: m.distance)

top_n = 40
sift_cc_vis = cv2.drawMatches(img1_gray, kp1_sift, img2_gray, kp2_sift, sift_cc_matches[:top_n], None, flags=2)
orb_cc_vis = cv2.drawMatches(img1_gray, kp1_orb, img2_gray, kp2_orb, orb_cc_matches[:top_n], None, flags=2)

fig, axes = plt.subplots(2, 1, figsize=(22, 14))
axes[0].imshow(sift_cc_vis, cmap='gray')
axes[0].set_title(f"SIFT Cross-Check Matches (Top {top_n})", color="#00cec9")
axes[0].axis("off")

axes[1].imshow(orb_cc_vis, cmap='gray')
axes[1].set_title(f"ORB Cross-Check Matches (Top {top_n})", color="#e17055")
axes[1].axis("off")

plt.tight_layout()
plt.show()

print("?? Baseline Parity Summary")
print(f"   SIFT keypoints: img1={len(kp1_sift)}, img2={len(kp2_sift)} | cross-check matches={len(sift_cc_matches)} | time={sift_time:.4f}s")
print(f"   ORB  keypoints: img1={len(kp1_orb)},  img2={len(kp2_orb)}  | cross-check matches={len(orb_cc_matches)}  | time={orb_time:.4f}s")


<a id="sec6"></a>
## 6 ? Camera Calibration, Stereo Vision & Depth

This section follows the baseline CV pipeline with a calibration/undistortion demonstration and a real IMC stereo depth example.


### 6.1 Chessboard Calibration & Undistortion

We include an explicit camera calibration demonstration (corner detection, intrinsic estimation, reprojection error, and undistortion) to complete the baseline CV workflow.


In [ ]:
# Synthetic chessboard calibration demo (Kaggle-safe, no external download)
chessboard_size = (9, 6)  # inner corners
sq = 50
rows, cols = chessboard_size[1] + 1, chessboard_size[0] + 1
board_h, board_w = rows * sq, cols * sq

board = np.zeros((board_h, board_w), dtype=np.uint8)
for r in range(rows):
    for c in range(cols):
        if (r + c) % 2 == 0:
            cv2.rectangle(board, (c * sq, r * sq), ((c + 1) * sq, (r + 1) * sq), 255, -1)

ret, corners = cv2.findChessboardCorners(board, chessboard_size, None)
print(f"?? Chessboard corners detected: {ret}")

if ret:
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
    corners_refined = cv2.cornerSubPix(board, corners, (11, 11), (-1, -1), criteria)

    objp = np.zeros((chessboard_size[0] * chessboard_size[1], 3), np.float32)
    objp[:, :2] = np.indices(chessboard_size).T.reshape(-1, 2)

    obj_points = [objp]
    img_points = [corners_refined]

    _, K_calib, dist_calib, rvecs, tvecs = cv2.calibrateCamera(
        obj_points, img_points, board.shape[::-1], None, None
    )

    proj, _ = cv2.projectPoints(objp, rvecs[0], tvecs[0], K_calib, dist_calib)
    reproj_err = cv2.norm(corners_refined, proj, cv2.NORM_L2) / len(proj)

    board_bgr = cv2.cvtColor(board, cv2.COLOR_GRAY2BGR)
    board_with_corners = cv2.drawChessboardCorners(board_bgr.copy(), chessboard_size, corners_refined, ret)
    board_undist = cv2.undistort(board_bgr, K_calib, dist_calib)

    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    axes[0].imshow(board, cmap='gray')
    axes[0].set_title('Synthetic Chessboard', color="#00cec9")
    axes[1].imshow(cv2.cvtColor(board_with_corners, cv2.COLOR_BGR2RGB))
    axes[1].set_title('Detected & Refined Corners', color="#fdcb6e")
    axes[2].imshow(cv2.cvtColor(board_undist, cv2.COLOR_BGR2RGB))
    axes[2].set_title('Undistorted View', color="#e17055")
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

    print("
?? Estimated Intrinsic Matrix K:")
    print(K_calib)
    print("
?? Distortion Coefficients:")
    print(dist_calib.ravel())
    print(f"
? Reprojection Error: {reproj_err:.6f}")
else:
    print("Chessboard detection failed; calibration demo skipped.")


### 6.2 Stereo Disparity & Depth on IMC Pair

Using a real Notre Dame stereo pair from IMC 2022 to estimate disparity and relative depth.


In [ ]:
# Use Notre Dame pair
ND_PATH1 = os.path.join(TRAIN_DIR, "notre_dame_front_facade/images/01496270_2526130741.jpg")
ND_PATH2 = os.path.join(TRAIN_DIR, "notre_dame_front_facade/images/01502738_4846283028.jpg")
left = cv2.imread(ND_PATH1, cv2.IMREAD_GRAYSCALE)
right = cv2.imread(ND_PATH2, cv2.IMREAD_GRAYSCALE)

# Resize to same dimensions for stereo matching
h = min(left.shape[0], right.shape[0])
w = min(left.shape[1], right.shape[1])
left = cv2.resize(left, (w, h))
right = cv2.resize(right, (w, h))

stereo = cv2.StereoBM_create(numDisparities=64, blockSize=15)
disparity = stereo.compute(left, right).astype(np.float32) / 16.0

# Depth from disparity
focal_length = 500; baseline = 0.1
depth = np.zeros_like(disparity)
valid = disparity > 0
depth[valid] = (focal_length * baseline) / disparity[valid]
depth_norm = cv2.normalize(depth, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(28, 6))
axes[0].imshow(left, cmap='gray'); axes[0].set_title("Left Image", color="#00cec9")
axes[1].imshow(right, cmap='gray'); axes[1].set_title("Right Image", color="#00cec9")
axes[2].imshow(disparity, cmap='magma'); axes[2].set_title("Disparity Map", color="#fdcb6e")
axes[3].imshow(depth_norm, cmap='plasma'); axes[3].set_title("Depth Map", color="#e17055")
for ax in axes: ax.axis("off")
fig.suptitle("Stereo Vision — Disparity & Depth on Real Dataset Images",
             fontsize=14, color="#00cec9")
plt.tight_layout(); plt.show()

print(f"\n📊 Stereo stats: disparity range [{disparity[valid].min():.1f}, {disparity[valid].max():.1f}]")
print(f"   Depth range [{depth[valid].min():.3f}, {depth[valid].max():.3f}]")

<a id="sec7"></a>
## 7 ? Fundamental & Essential Matrices ? Epipolar Geometry

### 7.1 SIFT-Based Epipolar Geometry


In [ ]:
# Use a Taj Mahal pair from IMC 2022 for SIFT-based epipolar geometry
scene = "taj_mahal"
pco_taj = train_map[scene]["pco_df"]
demo_row = pco_taj.iloc[850] if len(pco_taj) > 850 else pco_taj.iloc[0]
d_id1, d_id2 = demo_row.image_id_1, demo_row.image_id_2

d_img1 = cv2.imread(demo_row.image_path_1, cv2.IMREAD_GRAYSCALE)
d_img2 = cv2.imread(demo_row.image_path_2, cv2.IMREAD_GRAYSCALE)
d_rgb1 = cv2.cvtColor(cv2.imread(demo_row.image_path_1), cv2.COLOR_BGR2RGB)
d_rgb2 = cv2.cvtColor(cv2.imread(demo_row.image_path_2), cv2.COLOR_BGR2RGB)

plot_image_pair(demo_row.image_path_1, demo_row.image_path_2,
    [f"Taj Mahal — {d_id1} (COV={demo_row.covisibility:.3f})",
     f"Taj Mahal — {d_id2}"])

# Detect + Match + RANSAC
sift_d = cv2.SIFT_create(5000, contrastThreshold=-10000, edgeThreshold=-10000)
kp1d, des1d = sift_d.detectAndCompute(d_img1, None)
kp2d, des2d = sift_d.detectAndCompute(d_img2, None)
bf_d = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
matches_d = sorted(bf_d.match(des1d, des2d), key=lambda x: x.distance)

pts1d = np.float32([kp1d[m.queryIdx].pt for m in matches_d])
pts2d = np.float32([kp2d[m.trainIdx].pt for m in matches_d])

F_pred, mask_d = cv2.findFundamentalMat(pts1d, pts2d, cv2.USAC_MAGSAC, 0.25, 0.99999, 10000)
inl_mask = mask_d.ravel().astype(bool)

print(f"\n📊 Fundamental Matrix Estimation (Taj Mahal):")
print(f"   Matches: {len(matches_d)}, Inliers: {inl_mask.sum()}")
print(f"   F matrix:\n{F_pred}")

# Draw epipolar lines
pts1_inl = pts1d[inl_mask]; pts2_inl = pts2d[inl_mask]
lines1 = cv2.computeCorrespondEpilines(pts2_inl[:15].reshape(-1,1,2), 2, F_pred).reshape(-1,3)
epi1 = d_rgb1.copy(); epi2 = d_rgb2.copy()
for line, pt1, pt2 in zip(lines1, pts1_inl[:15], pts2_inl[:15]):
    c = tuple(np.random.randint(100, 255, 3).tolist())
    x0, y0 = 0, int(-line[2]/line[1])
    x1, y1 = d_img1.shape[1], int(-(line[2]+line[0]*d_img1.shape[1])/line[1])
    epi1 = cv2.line(epi1, (x0,y0), (x1,y1), c, 2)
    epi1 = cv2.circle(epi1, tuple(np.int32(pt1)), 8, c, -1)
    epi2 = cv2.circle(epi2, tuple(np.int32(pt2)), 8, c, -1)

fig, axes = plt.subplots(1, 2, figsize=(22, 8))
axes[0].imshow(epi1); axes[0].set_title("Epipolar Lines on Image 1", color="#00cec9")
axes[1].imshow(epi2); axes[1].set_title("Corresponding Points on Image 2", color="#00cec9")
for ax in axes: ax.axis("off")
fig.suptitle("Epipolar Geometry Visualization — Taj Mahal", fontsize=14, color="#fdcb6e")
plt.tight_layout(); plt.show()

# Essential matrix
K1 = id2cal[d_id1]["K"]; K2 = id2cal[d_id2]["K"]
E = K2.T @ F_pred @ K1
print(f"\n📊 Essential Matrix:\n{E}")


### 7.2 ORB + RANSAC Epipolar Geometry

This ORB block mirrors the baseline assignment flow and visualizes epipolar lines from ORB correspondences with RANSAC filtering.


In [ ]:
# ORB + RANSAC fundamental matrix and epipolar line visualization
orb_e = cv2.ORB_create(nfeatures=5000)

nd_path1 = os.path.join(TRAIN_DIR, "notre_dame_front_facade/images/01496270_2526130741.jpg")
nd_path2 = os.path.join(TRAIN_DIR, "notre_dame_front_facade/images/01502738_4846283028.jpg")
nd1 = cv2.imread(nd_path1, cv2.IMREAD_GRAYSCALE)
nd2 = cv2.imread(nd_path2, cv2.IMREAD_GRAYSCALE)
nd1_rgb = cv2.cvtColor(cv2.imread(nd_path1), cv2.COLOR_BGR2RGB)
nd2_rgb = cv2.cvtColor(cv2.imread(nd_path2), cv2.COLOR_BGR2RGB)

kp1_e, des1_e = orb_e.detectAndCompute(nd1, None)
kp2_e, des2_e = orb_e.detectAndCompute(nd2, None)

bf_e = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches_e = sorted(bf_e.match(des1_e, des2_e), key=lambda x: x.distance)

pts1_e = np.float32([kp1_e[m.queryIdx].pt for m in matches_e])
pts2_e = np.float32([kp2_e[m.trainIdx].pt for m in matches_e])

F_orb, m_orb = cv2.findFundamentalMat(pts1_e, pts2_e, cv2.FM_RANSAC)
if F_orb is None or F_orb.shape != (3, 3):
    raise RuntimeError("ORB fundamental matrix estimation failed")

inl = m_orb.ravel().astype(bool)
pts1_in = pts1_e[inl]
pts2_in = pts2_e[inl]

print("?? ORB + RANSAC Results")
print(f"   Total matches: {len(matches_e)}")
print(f"   Inliers: {inl.sum()}")
print(f"   Fundamental Matrix:
{F_orb}")

def draw_epilines(imgA_gray, imgB_gray, lines, ptsA, ptsB):
    h, w = imgA_gray.shape
    imgA = cv2.cvtColor(imgA_gray, cv2.COLOR_GRAY2RGB)
    imgB = cv2.cvtColor(imgB_gray, cv2.COLOR_GRAY2RGB)
    for line, pA, pB in zip(lines, ptsA, ptsB):
        color = tuple(np.random.randint(100, 255, 3).tolist())
        if abs(line[1]) < 1e-6:
            continue
        x0, y0 = 0, int(-line[2] / line[1])
        x1, y1 = w, int(-(line[2] + line[0] * w) / line[1])
        imgA = cv2.line(imgA, (x0, y0), (x1, y1), color, 2)
        imgA = cv2.circle(imgA, tuple(np.int32(pA)), 6, color, -1)
        imgB = cv2.circle(imgB, tuple(np.int32(pB)), 6, color, -1)
    return imgA, imgB

n_show = min(20, len(pts1_in))
lines_on_1 = cv2.computeCorrespondEpilines(pts2_in[:n_show].reshape(-1, 1, 2), 2, F_orb).reshape(-1, 3)
lines_on_2 = cv2.computeCorrespondEpilines(pts1_in[:n_show].reshape(-1, 1, 2), 1, F_orb).reshape(-1, 3)

epi1, pts_on_2 = draw_epilines(nd1, nd2, lines_on_1, pts1_in[:n_show], pts2_in[:n_show])
epi2, pts_on_1 = draw_epilines(nd2, nd1, lines_on_2, pts2_in[:n_show], pts1_in[:n_show])

fig, axes = plt.subplots(2, 2, figsize=(20, 14))
axes[0, 0].imshow(epi1)
axes[0, 0].set_title('Epipolar Lines on Image 1 (ORB)', color="#00cec9")
axes[0, 1].imshow(pts_on_2)
axes[0, 1].set_title('Corresponding Points on Image 2', color="#00cec9")
axes[1, 0].imshow(epi2)
axes[1, 0].set_title('Epipolar Lines on Image 2 (ORB)', color="#e17055")
axes[1, 1].imshow(pts_on_1)
axes[1, 1].set_title('Corresponding Points on Image 1', color="#e17055")
for ax in axes.ravel():
    ax.axis('off')
plt.tight_layout()
plt.show()


<a id="sec8"></a>
## 8 ? SfM Pipeline Overview & Key Takeaways

### Structure from Motion Pipeline
```
Raw Images ? Camera Calibration/Undistortion ? Feature Detection (SIFT/ORB/AKAZE)
? Feature Matching (BF/FLANN) ? RANSAC Outlier Rejection ? Fundamental Matrix
? Essential Matrix ? Pose Recovery (R, T) ? 3D Reconstruction
```

### Key Insights from this Notebook

| Finding | Detail |
|:--------|:-------|
| **Dataset** | 16 scenes, ~200-800 images each; images resized to ~800px longest edge |
| **Covisibility** | Threshold ? 0.1 recommended; varies significantly across scenes |
| **Calibration Demo** | Chessboard corner detection + intrinsic estimation + undistortion included |
| **SIFT** | Most robust classical detector; 128-D descriptors; strong geometric consistency |
| **ORB** | Fast detector with binary descriptors; effective with Hamming + RANSAC filtering |
| **AKAZE** | Non-linear scale space; handles texture variation well |
| **Stereo** | Disparity and depth visualization on real IMC pair included |
| **Epipolar Geometry** | Both SIFT-based and ORB-based epipolar line visualizations are included |
| **RANSAC** | Robust estimation is critical for reliable F-matrix recovery |

### ?? Next Steps
Notebook 2 builds the full production pipeline with cumulative evaluation (SIFT/LoFTR, matchers, robust estimators, and mAA).
